# 01: Data Exploration and Integration

**Objectives:**
- Load FAOSTAT and NASA POWER datasets
- Merge datasets by year
- Validate data coverage and identify missing values
- Generate initial descriptive statistics

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

## 1.1 Load FAOSTAT Datasets

In [ ]:
# Load raw data files
raw_dir = Path('data/raw')

# Load FAOSTAT yield data
yield_df = pd.read_csv(raw_dir / 'faostat_sri_lanka_rice_yield.csv')
print("Yield Data Shape:", yield_df.shape)
print("\nFirst few rows:")
print(yield_df.head())
print("\nYield Data Info:")
print(yield_df.info())

In [ ]:
# Load crop production data
crop_df = pd.read_csv(raw_dir / 'Crop_data.csv')
print("Crop Data Shape:", crop_df.shape)
print("\nUnique Elements:", crop_df['Element'].unique())
print("Unique Years:", sorted(crop_df['Year Code'].unique()))
print("\nArea harvested (Rice):")
area_df = crop_df[(crop_df['Element'] == 'Area harvested') & (crop_df['Item'] == 'Rice')]
print(area_df[['Year Code', 'Value']].head(10))

In [ ]:
# Load fertilizer data
fertilizer_df = pd.read_csv(raw_dir / 'Fertilizer_data.csv')
print("Fertilizer Data Shape:", fertilizer_df.shape)
print("\nUnique nutrients:", fertilizer_df['Item'].unique())
print("\nYear range:", fertilizer_df['Year Code'].min(), "-", fertilizer_df['Year Code'].max())
print("\nFertilizer totals by year:")
fert_by_year = fertilizer_df.groupby('Year Code')['Value'].sum()
print(fert_by_year)

In [ ]:
# Load irrigation data
irrigation_df = pd.read_csv(raw_dir / 'Irrigation_data.csv')
print("Irrigation Data Shape:", irrigation_df.shape)
print("\nIrrigation data:")
print(irrigation_df[['Year Code', 'Item', 'Value']].head(10))

## 1.2 Load NASA POWER Climate Data

In [ ]:
# Load NASA climate data
nasa_df = pd.read_csv(raw_dir / 'nasa_power_sri_lanka_monthly.csv')
print("NASA Data Shape:", nasa_df.shape)
print("\nColumns:", nasa_df.columns.tolist())
print("\nDate range:", nasa_df['date'].min(), "-", nasa_df['date'].max())
print("\nFirst few rows:")
print(nasa_df.head())
print("\nClimate variable ranges:")
print(nasa_df[['temperature_c', 'precip_mm_day', 'solar_mj_m2_day']].describe())

## 1.3 Aggregate Climate Data to Annual Level

In [ ]:
# Extract year from date and aggregate
nasa_df['year'] = nasa_df['date'].astype(str).str[:4].astype(int)

# Annual climate aggregates
annual_climate = nasa_df.groupby('year').agg({
    'temperature_c': 'mean',
    'precip_mm_day': 'mean',
    'solar_mj_m2_day': 'mean'
}).reset_index()

annual_climate.columns = ['year', 'temp_annual_mean', 'precip_annual_mean', 'solar_annual_mean']
print("Annual Climate Data:")
print(annual_climate)

## 1.4 Merge Datasets by Year

In [ ]:
# Prepare yield data
yield_clean = yield_df.rename(columns={'year': 'year'}).copy()
yield_clean['year'] = yield_clean['year'].astype(int)
yield_clean['yield_kg_ha'] = yield_clean['yield_hg_ha'] * 10  # Convert hectograms to kg
yield_clean = yield_clean[['year', 'yield_kg_ha']]

# Prepare area data
area_clean = area_df[area_df['Item'] == 'Rice'].copy()
area_clean = area_clean.rename(columns={'Year Code': 'year', 'Value': 'area_harvested_ha'})
area_clean = area_clean[['year', 'area_harvested_ha']].drop_duplicates()
area_clean['year'] = area_clean['year'].astype(int)

# Prepare fertilizer data
fert_clean = fertilizer_df.groupby('Year Code').agg({'Value': 'sum'}).reset_index()
fert_clean.columns = ['year', 'fertilizer_tonnes']
fert_clean['year'] = fert_clean['year'].astype(int)

# Prepare irrigation data
irrig_clean = irrigation_df[irrigation_df['Item'].str.contains('irrigation', case=False, na=False)].copy()
irrig_clean = irrig_clean.rename(columns={'Year Code': 'year', 'Value': 'irrigation_area_1000ha'})
irrig_clean['irrigation_area_ha'] = irrig_clean['irrigation_area_1000ha'] * 1000
irrig_clean = irrig_clean[['year', 'irrigation_area_ha']].drop_duplicates()
irrig_clean['year'] = irrig_clean['year'].astype(int)

# Merge all datasets
merged_df = yield_clean.merge(area_clean, on='year', how='outer')
merged_df = merged_df.merge(fert_clean, on='year', how='outer')
merged_df = merged_df.merge(irrig_clean, on='year', how='outer')
merged_df = merged_df.merge(annual_climate, on='year', how='outer')

# Sort by year
merged_df = merged_df.sort_values('year').reset_index(drop=True)

print("Merged Dataset Shape:", merged_df.shape)
print("\nData Coverage:")
print(merged_df)

## 1.5 Data Quality Assessment

In [ ]:
# Check missing values
print("Missing Values:")
print(merged_df.isnull().sum())
print("\nMissing Percentage:")
print((merged_df.isnull().sum() / len(merged_df) * 100).round(2))

In [ ]:
# Descriptive statistics
print("\nDescriptive Statistics:")
print(merged_df.describe())

## 1.6 Preliminary Visualizations

In [ ]:
# Time series plots
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Yield trend
axes[0, 0].plot(merged_df['year'], merged_df['yield_kg_ha'], marker='o', linewidth=2)
axes[0, 0].set_title('Rice Yield Over Time', fontsize=12, fontweight='bold')
axes[0, 0].set_xlabel('Year')
axes[0, 0].set_ylabel('Yield (kg/ha)')
axes[0, 0].grid(True, alpha=0.3)

# Temperature trend
axes[0, 1].plot(merged_df['year'], merged_df['temp_annual_mean'], marker='s', color='red', linewidth=2)
axes[0, 1].set_title('Average Annual Temperature', fontsize=12, fontweight='bold')
axes[0, 1].set_xlabel('Year')
axes[0, 1].set_ylabel('Temperature (°C)')
axes[0, 1].grid(True, alpha=0.3)

# Precipitation trend
axes[1, 0].plot(merged_df['year'], merged_df['precip_annual_mean'], marker='^', color='blue', linewidth=2)
axes[1, 0].set_title('Average Daily Precipitation', fontsize=12, fontweight='bold')
axes[1, 0].set_xlabel('Year')
axes[1, 0].set_ylabel('Precipitation (mm/day)')
axes[1, 0].grid(True, alpha=0.3)

# Fertilizer usage
axes[1, 1].bar(merged_df['year'], merged_df['fertilizer_tonnes'], color='green', alpha=0.7)
axes[1, 1].set_title('Annual Fertilizer Use', fontsize=12, fontweight='bold')
axes[1, 1].set_xlabel('Year')
axes[1, 1].set_ylabel('Fertilizer (tonnes)')
axes[1, 1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('results/graphs/01_time_series_overview.png', dpi=300, bbox_inches='tight')
plt.show()

print("Saved: results/graphs/01_time_series_overview.png")

## Summary

- **Yield Data**: {0} years of data (2000-2024)
- **Climate Data**: Complete monthly coverage for analysis period
- **Farm Inputs**: Fertilizer and irrigation data available (2020-2023)
- **Next Steps**: Clean missing values, engineer monitoring_index, and prepare for analysis